# Μάθημα 5: Retrieval-Augmented Generation (RAG)

Πώς δίνουμε σε ένα LLM τη δυνατότητα να διαβάσει τα ιδιωτικά μας PDF (π.χ. τα συμβόλαια μιας εταιρείας) και να απαντάει ερωτήσεις αποκλειστικά πάνω σε αυτά;

Καλώς ήρθατε στο **RAG**. Αντί να στείλουμε ένα βιβλίο 1000 σελίδων στο ChatGPT (το οποίο είναι ακριβό και πολλές φορές αδύνατο λόγω του Context Limit), κάνουμε τα εξής:
1. Κόβουμε το βιβλίο σε παραγράφους (Chunks).
2. Φτιάχνουμε Embeddings για την κάθε παράγραφο και τα αποθηκεύουμε σε μια Vector Database (Bάση Δεδομένων Διανυσμάτων).
3. Όταν ο χρήστης κάνει μια ερώτηση, ψάχνουμε στη βάση τις 3 πιο σχετικές παραγράφους.
4. Δίνουμε ΜΟΝΟ αυτές τις 3 παραγράφους στο LLM και του λέμε: "Με βάση αυτό το κείμενο, απάντα στην ερώτηση".

## Προσομοίωση ενός RAG Pipeline με LangChain & FAISS
Εδώ θα χρησιμοποιήσουμε το `LangChain` για να ενορχηστρώσουμε τη διαδικασία.

In [ ]:
from langchain.text_splitter import RecursiveCharacterTextSplitter
import numpy as np

print("Βήμα 1: Το έγγραφό μας (Ιδιωτικά Δεδομένα)")
private_doc = """
Η εταιρεία 'AI-Corp' ιδρύθηκε το 2025 από τον Δημήτρη Καρύδα.
Το βασικό προϊόν της εταιρείας είναι ένα RAG σύστημα για νομικά γραφεία.
Τα κεντρικά γραφεία βρίσκονται στην Αθήνα, και ο τζίρος το 2026 έφτασε το 1 εκατομμύριο ευρώ.
"""

print("Βήμα 2: Text Splitting (Κόβουμε το κείμενο σε Chunks)")
# Κόβουμε ανά 60 χαρακτήρες, κρατώντας ένα overlap 10 χαρακτήρων για να μη χάνουμε το νόημα
splitter = RecursiveCharacterTextSplitter(chunk_size=60, chunk_overlap=10)
chunks = splitter.split_text(private_doc.strip())

for i, chunk in enumerate(chunks):
    print(f"  [Chunk {i+1}]: {chunk}")

## Ανάκτηση (Retrieval) και Ερώτηση
Σε ένα πραγματικό σενάριο, τα chunks θα γίνονταν Embeddings (μέσω του OpenAI API π.χ.) και θα έμπαιναν σε μια βάση FAISS. Όταν ρωτάμε 'Πού είναι τα κεντρικά;', το σύστημα κάνει μαθηματική αναζήτηση.

In [ ]:
# Προσομοίωση του Retrieval (Η βάση βρήκε το Chunk 3 ως το πιο σχετικό με τη λέξη 'γραφεία' ή 'Αθήνα')
user_query = "Πού βρίσκονται τα γραφεία της εταιρείας;"
retrieved_chunk = chunks[2] # "Τα κεντρικά γραφεία βρίσκονται στην Αθήνα..."

print(f"\nΕρώτηση Χρήστη: '{user_query}'")
print(f"Βρέθηκε Σχετικό Έγγραφο (Retrieval): '{retrieved_chunk}'")

print("\n--- Το τελικό Prompt που πάει στο ChatGPT ---")
final_prompt = f"""
Είσαι ένας βοηθός AI. Απάντησε στην ερώτηση βασιζόμενος ΜΟΝΟ στο παρακάτω πλαίσιο πληροφοριών.

Πλαίσιο:
{retrieved_chunk}

Ερώτηση: {user_query}
Απάντηση:
"""
print(final_prompt)

print("Το LLM τώρα θα απαντήσει τέλεια: 'Τα γραφεία βρίσκονται στην Αθήνα.' Χωρίς καμία παραίσθηση (Hallucination)!")